# lstm_sgd_canevas

**Auteurs :** Hugo Morlet, Naël Bensaadi
**UFR MIM — Université de Lorraine — 2026**

Réseau récurrent LSTM (codé entièrement avec NumPy) entraîné par descente de
gradient stochastique avec :

- décroissance du *learning rate* (LR decay)
- rognage du gradient (*gradient clipping*)
- propagation arrière dans le temps (BPTT)

## 1 Réseau récurrent pour générer du texte

Ce réseau repose sur des unités **LSTM**. À l'aide d'un texte fourni en entrée,
il apprend des mots et les relations de proximité entre ces mots.

Une fois l'apprentissage fait, la *prédiction* consiste à générer un texte à
l'aide des premiers mots du texte appris. Il n'y a aucun critère de grammaire
ni de conjugaison.

### 1.1 Prétraitements

Avant la phase d'apprentissage, le texte fourni en entrée est transformé en
une liste d'entiers. Ces entiers correspondent aux indexes dans le
dictionnaire des mots utilisés (et uniquement ceux-ci).
Le dictionnaire est donc construit en premier.

Puis une table est construite qui met les mots en correspondance avec les
indexes. La table inverse qui met en correspondance les indexes avec les mots.

Il y a alors trois problèmes majeurs à surmonter :

* Ces tables de correspondance ne disent rien des relations de proximité entre
  les mots du texte.
* Les réseaux connexionnistes ne savent manipuler que des valeurs réelles.
* Le texte sous forme de liste est de très (trop) grande taille.

Pour répondre à ces soucis, on transforme chaque index (chaque mot donc) en un
*vecteur englobant*. Ces vecteurs sont initialisés aléatoirement et leurs
coefficients sont corrigés à chaque étape de l'apprentissage comme n'importe
quel poids. L'initialisation suit la loi normale :

$$ \mathcal{N}(0, \sigma) \quad \text{avec} \quad \sigma = \sqrt{\frac{2}{|\mathcal{V}| + |\mathcal{E}|}} $$

où $\mathcal{V}$ est l'espace des indexes/mots et $\mathcal{E}$ est un espace
de dimension moindre.

L'ensemble des vecteurs est regroupé en une matrice englobante $E$ de taille
$|\mathcal{V}| \times |\mathcal{E}|$. La matrice $E$ effectue une projection
d'un espace dans un espace *réduit*. Chaque ligne est le vecteur englobant de
l'index d'un mot du dictionnaire.

Quand deux vecteurs sont similaires dans leurs coefficients cela signifie que
leurs mots partagent des contextes similaires. Par exemple les phrases
"Ce chien est mignon" et "Ce chat est mignon" sont des contextes similaires
pour les mots *chien* et *chat*, et non parce qu'ils ont des lettres en commun.

### 1.2 L'apprentissage

Le réseau est assez basique puisqu'il est constitué d'une seule couche cachée
dont les unités sont des LSTM.

L'apprentissage prend comme données une séquence de mots et comme cible la
même séquence mais avec un décalage d'un mot. Les deux séquences ont la même
longueur et chaque temps $t$ le long de ces séquences le mot cible est le mot
à prédire à partir du mot donné.

La rétropropagation commence par une activation **Softmax** et le calcul de
l'**entropie croisée** comme vous l'avez déjà utilisé. Ensuite tous les
gradients à chaque temps $t$ le long de la séquence sont calculés en
commençant par ceux de la sortie ($\partial L/\partial W_{out}$ et
$\partial L/\partial B_{out}$).

N'oubliez pas que les poids sont les mêmes à chaque temps mais qu'il faut
cumuler leurs gradients au fil du temps.

### 1.3 Coder le RNN-LSTM

Il s'agit d'un code à trous, on a complété les trous (marqués par `...` dans
le canevas du sujet) en suivant la structure imposée.

> **Note d'implémentation** : la cellule LSTM ci-dessous garde un attribut
> `self.cache` qui est écrasé à chaque appel de `forward`. Or pour faire la
> rétropropagation à travers le temps, on a besoin du cache de **chaque** pas
> de temps. On le résout en stockant la liste de tous les caches dans le
> réseau (`self.cache["cell_caches"]`) au moment du forward, puis en
> remettant le bon cache dans la cellule avant chaque appel à `backward`.

In [ ]:
import numpy as np

#### 1.3.1 Cellule LSTM

Une cellule LSTM combine quatre "unités" :

| Unité | Rôle | Activation |
|---|---|---|
| $i_t$ | porte d'entrée — ce qu'on ajoute à la mémoire | sigmoid |
| $f_t$ | porte d'oubli — ce qu'on garde de la mémoire passée | sigmoid |
| $o_t$ | porte de sortie — ce qui sort vers $h_t$ | sigmoid |
| $\tilde{c}_t$ | candidat — nouvelle info potentielle | tanh |

Les équations :

$$
\begin{aligned}
i_t &= \sigma(x_t W_i + h_{t-1} U_i + B_i) \\
f_t &= \sigma(x_t W_f + h_{t-1} U_f + B_f) \\
o_t &= \sigma(x_t W_o + h_{t-1} U_o + B_o) \\
\tilde{c}_t &= \tanh(x_t W_c + h_{t-1} U_c + B_c) \\
c_t &= f_t \odot c_{t-1} + i_t \odot \tilde{c}_t \\
h_t &= o_t \odot \tanh(c_t)
\end{aligned}
$$

où $\odot$ est le produit terme à terme.

In [ ]:
class LSTMCell:

    def __init__(self, input_size, hidden_size):
        self.input_size = input_size
        self.hidden_size = hidden_size

        # Initialisation Xavier / Glorot :
        #   on tire dans une loi normale N(0, sigma^2) avec
        #   sigma = sqrt( 2 / (n_entree + n_sortie) )
        # Objectif : que la variance des signaux reste stable entre couches
        # (évite que les activations explosent ou s'éteignent au début).
        scale = np.sqrt(2.0 / (input_size + hidden_size))

        # Pour chaque "unité" on a une matrice W (poids sur x_t),
        # une matrice U (poids sur h_{t-1}) et un biais B (initialisé à 0).
        # W : |E|x|H|, U : |H|x|H|, B : |H|.

        # Porte d'entrée (input)
        self.Wi = np.random.randn(input_size, hidden_size) * scale
        self.Ui = np.random.randn(hidden_size, hidden_size) * scale
        self.Bi = np.zeros(hidden_size)

        # Porte d'oubli (forget)
        self.Wf = np.random.randn(input_size, hidden_size) * scale
        self.Uf = np.random.randn(hidden_size, hidden_size) * scale
        self.Bf = np.zeros(hidden_size)

        # Porte de sortie (output)
        self.Wo = np.random.randn(input_size, hidden_size) * scale
        self.Uo = np.random.randn(hidden_size, hidden_size) * scale
        self.Bo = np.zeros(hidden_size)

        # Candidat de mémoire (cell candidate)
        self.Wc = np.random.randn(input_size, hidden_size) * scale
        self.Uc = np.random.randn(hidden_size, hidden_size) * scale
        self.Bc = np.zeros(hidden_size)

        # Mémoire des valeurs intermédiaires (réutilisées dans backward)
        self.cache = {}

    def sigmoid(self, x):
        # Garde-fou numérique : np.exp(-x) devient énorme pour x très négatif
        x = np.clip(x, -500, 500)
        return 1 / (1 + np.exp(-x))

    def tanh(self, x):
        x = np.clip(x, -500, 500)
        return np.tanh(x)

    def forward(self, x_t, h_prev, c_prev):
        # Trois portes + un candidat
        i_t = self.sigmoid(x_t @ self.Wi + h_prev @ self.Ui + self.Bi)
        f_t = self.sigmoid(x_t @ self.Wf + h_prev @ self.Uf + self.Bf)
        o_t = self.sigmoid(x_t @ self.Wo + h_prev @ self.Uo + self.Bo)
        c_tilde = self.tanh(x_t @ self.Wc + h_prev @ self.Uc + self.Bc)

        # Mise à jour de la mémoire : on oublie une partie de c_{t-1} et
        # on ajoute une partie du nouveau candidat.
        c_t = f_t * c_prev + i_t * c_tilde

        # On mémorise les valeurs intermédiaires pour le backward
        # On renvoie l'état et la cellule cachées
        h_t = o_t * self.tanh(c_t)

        self.cache = {
            "x_t": x_t, "h_prev": h_prev, "c_prev": c_prev,
            "i_t": i_t, "f_t": f_t, "o_t": o_t,
            "c_tilde": c_tilde, "c_t": c_t, "h_t": h_t,
        }
        return h_t, c_t

    def backward(self, dh_t, dc_t):
        # En argument on a dL/dh_t et dL/dc_{t+1}
        # Et l'on retrouve la mémoire :))
        cache = self.cache
        x_t = cache["x_t"]
        h_prev = cache["h_prev"]
        c_prev = cache["c_prev"]
        i_t = cache["i_t"]
        f_t = cache["f_t"]
        o_t = cache["o_t"]
        c_tilde = cache["c_tilde"]
        c_t_val = cache["c_t"]

        # tanh(c_t) revient plusieurs fois : on le calcule une seule fois
        tanh_c_t = np.tanh(c_t_val)

        # --- Étape 1 : à travers h_t = o_t * tanh(c_t) (règle du produit) ---
        dtanh = dh_t * o_t                  # dL/d(tanh(c_t))
        do_t = dh_t * tanh_c_t              # dL/d(o_t)  (valeur APRÈS sigmoid)

        # --- Étape 2 : gradient TOTAL sur c_t ---
        # c_t a 2 sorties : (a) via tanh(c_t) -> h_t  (b) directement vers c_{t+1}
        # Donc on cumule les deux contributions :
        dc_t_total = dtanh * (1 - tanh_c_t ** 2) + dc_t

        # --- Étape 3 : à travers c_t = f_t * c_prev + i_t * c_tilde ---
        dc_tilde = dc_t_total * i_t         # dL/d(c_tilde)  (APRÈS tanh)
        df_t = dc_t_total * c_prev          # dL/d(f_t)      (APRÈS sigmoid)
        di_t = dc_t_total * c_tilde         # dL/d(i_t)      (APRÈS sigmoid)

        # Gradient transmis vers le passé (cellule t-1)
        dc_prev = dc_t_total * f_t

        # --- Étape 4 : à travers les activations (passe en "raw" = avant activation) ---
        # sigmoid : g' = g * (1 - g)
        # tanh    : g' = 1 - g^2
        di_raw = di_t * i_t * (1 - i_t)
        df_raw = df_t * f_t * (1 - f_t)
        do_raw = do_t * o_t * (1 - o_t)
        dc_tilde_raw = dc_tilde * (1 - c_tilde ** 2)

        # --- Étape 5 : gradients sur les poids ---
        # ATTENTION. Produit externe : M = V_1 V_2^T
        dWi = np.outer(x_t, di_raw)
        dUi = np.outer(h_prev, di_raw)
        dBi = di_raw.copy()

        dWf = np.outer(x_t, df_raw)
        dUf = np.outer(h_prev, df_raw)
        dBf = df_raw.copy()

        dWo = np.outer(x_t, do_raw)
        dUo = np.outer(h_prev, do_raw)
        dBo = do_raw.copy()

        dWc = np.outer(x_t, dc_tilde_raw)
        dUc = np.outer(h_prev, dc_tilde_raw)
        dBc = dc_tilde_raw.copy()

        # dL/dx_t : on somme les contributions des 4 unités
        dx_t = (di_raw @ self.Wi.T + df_raw @ self.Wf.T
                + do_raw @ self.Wo.T + dc_tilde_raw @ self.Wc.T)

        # dL/dh_{t-1} : idem
        dh_prev = (di_raw @ self.Ui.T + df_raw @ self.Uf.T
                   + do_raw @ self.Uo.T + dc_tilde_raw @ self.Uc.T)

        # On renvoie le dictionnaire des gradients
        return {
            "dWi": dWi, "dUi": dUi, "dBi": dBi,
            "dWf": dWf, "dUf": dUf, "dBf": dBf,
            "dWo": dWo, "dUo": dUo, "dBo": dBo,
            "dWc": dWc, "dUc": dUc, "dBc": dBc,
            "dx_t": dx_t, "dh_prev": dh_prev, "dc_prev": dc_prev,
        }

#### 1.3.2 Réseau neuronal récurrent

Avec une unique couche LSTM.

Le réseau enchaîne :

1. une couche d'**embedding** $E$ (taille $|V| \times |E|$),
2. la cellule LSTM ci-dessus,
3. une couche de **sortie** linéaire (taille $|H| \times |V|$) qui produit les
   logits, puis softmax pour obtenir des probabilités.

In [ ]:
class LSTMNetwork:

    def __init__(self, vocab_size, embed_size, hidden_size):
        self.vocab_size = vocab_size
        self.embed_size = embed_size
        self.hidden_size = hidden_size

        # Init Xavier/Glorot pour l'embedding |V| -> |E|
        # La matrice englobante de taille |V|x|E| suit une loi normale
        # avec un écart-type adapté au corpus et à l'englobant
        scale = np.sqrt(2.0 / (vocab_size + embed_size))
        self.E = np.random.randn(vocab_size, embed_size) * scale

        # Unique couche récurrente
        self.lstm = LSTMCell(embed_size, hidden_size)

        # (1) Init Xavier/Glorot pour la couche de sortie |H| -> |V|
        # (2) Les poids en sortie suivent une loi normale
        # (3) Les biais sont initialisés à zéro
        # (4) Préparation du cache (de la mémoire)
        scale = np.sqrt(2.0 / (hidden_size + vocab_size))
        self.Wout = np.random.randn(hidden_size, vocab_size) * scale
        self.Bout = np.zeros(vocab_size)

        self.cache = {}

    def forward_sequence(self, x_seq):
        # Initialiser
        #   les états cachés,
        #   les cellules cachées,
        #   la mémoire des états cachés de chaque mot,
        #   les logits
        h = np.zeros(self.hidden_size)
        c = np.zeros(self.hidden_size)
        h_seq = []
        logits = np.empty((len(x_seq), self.vocab_size))

        # On garde aussi la liste des caches de la cellule à chaque pas,
        # nécessaire pour faire la BPTT plus tard.
        cell_caches = []

        # Pour chaque temps t associé à la séquence x_seq :
        #   (a) Le mot (entier) est transformé en son vecteur englobant
        #   (b) La phase avant est faite pour ce vecteur englobant
        #   (c) Le nouvel historique est mémorisé
        #   (d) Le logit est calculé avec les poids et biais de sortie
        for t, x_t in enumerate(x_seq):
            e_t = self.E[x_t]
            h, c = self.lstm.forward(e_t, h, c)
            cell_caches.append(self.lstm.cache)
            h_seq.append(h.copy())
            logits[t, :] = h @ self.Wout + self.Bout

        self.cache = {
            "x_seq": x_seq,
            "h_seq": h_seq,
            "logits": logits,
            "cell_caches": cell_caches,
        }
        return logits

    def softmax(self, logits):
        exp_logits = np.exp(logits - np.max(logits, axis=-1, keepdims=True))
        return exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)

    def cross_entropy_loss(self, logits, targets):
        n = len(targets)

        if logits.shape[0] != n:
            raise ValueError(
                f"Shape mismatch: logits has {logits.shape[0]} positions, "
                f"but targets has {n} elements"
            )

        probs = self.softmax(logits)
        selected_probs = probs[np.arange(n), targets]
        # On évite les valeurs trop faibles pouvant dégénérer dans log(0)
        selected_probs[selected_probs < 1e-8] = 1e-8

        return -np.mean(np.log(selected_probs))

    def backward_sequence(self, targets, clip_norm=5.0):
        cache = self.cache
        logits = cache["logits"]
        h_seq = cache["h_seq"]
        x_seq = cache["x_seq"]
        cell_caches = cache["cell_caches"]
        n = len(targets)

        # 1. On effectue le softmax des logits
        # 2. On calcule le coût en sortie :  dlogits = (p_i - y_i) / n
        #    où o sont les logits, y est un one-hot |V|
        probs = self.softmax(logits)
        dlogits = probs.copy()
        dlogits[np.arange(n), targets] -= 1
        dlogits /= n

        # dL/dWout de taille |H|x|V| est le cumul, à chaque temps t,
        # des produits des états cachés (h_seq[t]) par les coûts en sortie (dlogits[t]).
        # ATTENTION : il s'agit de listes et non de vecteurs : le produit au temps
        # t est une matrice |H|x|V|.
        # dL/dBout est le cumul dans le temps des coûts en sortie dlogits[t].
        dWout = np.zeros_like(self.Wout)
        for t in range(n):
            dWout += np.outer(h_seq[t], dlogits[t])
        dBout = np.sum(dlogits, axis=0)

        # On initialise les prochains états et cellules cachées (depuis le futur)
        dh_next = np.zeros(self.hidden_size)
        dc_next = np.zeros(self.hidden_size)

        # On initialise le dictionnaire des gradients
        total_grads = {
            "dE": np.zeros_like(self.E),
            "dWout": dWout,
            "dBout": dBout,
            "dWi": np.zeros_like(self.lstm.Wi),
            "dUi": np.zeros_like(self.lstm.Ui),
            "dBi": np.zeros_like(self.lstm.Bi),
            "dWf": np.zeros_like(self.lstm.Wf),
            "dUf": np.zeros_like(self.lstm.Uf),
            "dBf": np.zeros_like(self.lstm.Bf),
            "dWo": np.zeros_like(self.lstm.Wo),
            "dUo": np.zeros_like(self.lstm.Uo),
            "dBo": np.zeros_like(self.lstm.Bo),
            "dWc": np.zeros_like(self.lstm.Wc),
            "dUc": np.zeros_like(self.lstm.Uc),
            "dBc": np.zeros_like(self.lstm.Bc),
        }

        # On effectue le voyage vers le passé
        for t in range(len(x_seq) - 1, -1, -1):
            dh_t = dlogits[t] @ self.Wout.T + dh_next      # dL/dh_t = dL/o * Wout^T + (h, t+1)
            dc_t = dc_next                                  # (c, t+1)

            # On remet le bon cache dans la cellule (sinon : seulement le dernier)
            self.lstm.cache = cell_caches[t]

            # On effectue une rétropropagation
            grads = self.lstm.backward(dh_t, dc_t)

            # On cumule les correctifs des poids et des biais
            for key in ("dWi", "dUi", "dBi", "dWf", "dUf", "dBf",
                        "dWo", "dUo", "dBo", "dWc", "dUc", "dBc"):
                total_grads[key] += grads[key]
            total_grads["dE"][x_seq[t]] += grads["dx_t"]

            # On récupère les gradients précédents
            dh_next = grads["dh_prev"]
            dc_next = grads["dc_prev"]

        # On effectue une coupure des gradients trop forts (clipping)
        if clip_norm is not None:
            total_norm = 0
            for key in total_grads:
                if isinstance(total_grads[key], np.ndarray):
                    total_norm += np.sum(total_grads[key] ** 2)
            total_norm = np.sqrt(total_norm)

            if total_norm > clip_norm:
                for key in total_grads:
                    if isinstance(total_grads[key], np.ndarray):
                        total_grads[key] *= clip_norm / total_norm

        return total_grads

#### 1.3.3 Algorithme d'optimisation

Une descente de gradient stochastique *basique* avec un taux d'apprentissage
décroissant :

$$ lr(\text{epoch}) = \frac{lr_0}{1 + \text{decay} \times \text{epoch}} $$

Au début on apprend vite, plus tard on ajuste finement.

In [ ]:
class SGDOptimizer:
    def __init__(self, lr=0.01, decay_rate=0.01):
        self.initial_lr = lr
        self.decay_rate = decay_rate
        self.epoch = 0

    def step(self):
        self.epoch += 1

    def get_lr(self):  # le pas diminue avec le temps
        return self.initial_lr / (1 + self.decay_rate * self.epoch)

    def update(self, params, grads):
        current_lr = self.get_lr()
        for key in params:
            grad_key = "d" + key
            params[key] -= current_lr * grads[grad_key]

#### 1.3.4 Phase d'apprentissage

À chaque epoch on parcourt le corpus en sous-séquences de longueur
`seq_length`. Pour chaque sous-séquence on fait un forward, on calcule la
perte, on fait un backward (avec clipping), puis on met à jour les poids.

In [ ]:
def train_text(model, optimizer, data, seq_length=10, epochs=100, verbose_every=10):
    for epoch in range(epochs):
        total_loss = 0
        n_batches = 0

        for start in range(0, len(data) - seq_length, seq_length):
            x_seq = data[start:start + seq_length]
            y_seq = data[start + 1:start + seq_length + 1]

            logits = model.forward_sequence(x_seq)
            loss = model.cross_entropy_loss(logits, y_seq)
            total_loss += loss
            n_batches += 1

            grads = model.backward_sequence(y_seq, clip_norm=5.0)

            # On construit le dictionnaire des poids
            # Pour les gradients il suffira d'ajouter "d" en tête de ces clés
            params = {
                "E": model.E,
                "Wout": model.Wout,
                "Bout": model.Bout,
                "Wi": model.lstm.Wi, "Ui": model.lstm.Ui, "Bi": model.lstm.Bi,
                "Wf": model.lstm.Wf, "Uf": model.lstm.Uf, "Bf": model.lstm.Bf,
                "Wo": model.lstm.Wo, "Uo": model.lstm.Uo, "Bo": model.lstm.Bo,
                "Wc": model.lstm.Wc, "Uc": model.lstm.Uc, "Bc": model.lstm.Bc,
            }
            optimizer.update(params, grads)

        optimizer.step()
        avg_loss = total_loss / n_batches if n_batches > 0 else 0  # on ne sait jamais
        current_lr = optimizer.get_lr()

        if epoch % verbose_every == 0:
            print(f"Epoch {epoch}: Loss = {avg_loss:.4f}, LR = {current_lr:.6f}")

    return data

#### 1.3.5 Construction de la base de données

Petit corpus en anglais répété 50 fois (ou bien la préface anglaise des Trois
Mousquetaires si on décommente le bloc `open(...)`). On en extrait :

1. la liste des mots,
2. un dictionnaire mot -> indice,
3. le dictionnaire inverse indice -> mot,
4. la version codée du texte (liste d'entiers).

In [ ]:
def load_complete_data():
    """
    1. Un corpus que l'on répète 50 fois pour qu'il soit conséquent
       Ou bien la préface en anglais du livre Les Trois Mousquetaires
    2. Découpage du corpus en mots
    3. Un dico de clefs où
       3.1 les clefs sont les mots
       3.2 les valeurs sont les indices de ces mots dans l'ensemble des mots
           utilisés par le corpus (sans répétition)
    4. Un dico inverse de clefs les indices et de valeurs les mots
    5. Le corpus est alors codé par une liste d'entiers
    """
    text = """
    the new model that the company announced yesterday was better
    than expected. investors were pleased with the results and the stock price
    went up significantly. the market responded positively to the announcement
    and trading volume increased. analysts believe that the company will
    continue to grow in the coming quarters. the board of directors approved the
    new strategy unanimously during the meeting. shareholders expressed
    satisfaction with the quarterly earnings report released today. the chief
    executive officer stated that innovation remains the top priority.
    competition in the industry has intensified but the company maintains its
    position. revenue growth exceeded expectations for the third consecutive
    quarter this year. cost reduction initiatives have improved operational
    efficiency across all divisions.
    """ * 50

    # with open("Les_Trois_Mousquetaires_Echantillon.txt", "r") as f:
    #     text = f.read()

    words = text.lower().split()
    word_to_idx = {w: i for i, w in enumerate(set(words))}
    idx_to_word = {i: w for w, i in word_to_idx.items()}
    data = [word_to_idx[w] for w in words]
    return data, word_to_idx, idx_to_word

#### 1.3.6 Génération de texte

Après l'apprentissage on prend 5 mots de "graine" et on prédit 20 mots à la
suite. À chaque étape on garde une fenêtre de 10 mots maximum et on prend
l'`argmax` du dernier logit (génération *gloutonne*, sans aléa).

In [ ]:
print("=" * 75)
print("UN RÉSEAU LSTM BASIQUE - GÉNÉRATION DE TEXTE + LR Decay + GRADIENT CLIPPING")
print("=" * 75)

# Reproductibilité (résultats identiques d'une exécution à l'autre)
np.random.seed(42)

data, word_to_idx, idx_to_word = load_complete_data()
vocab_size = len(word_to_idx)

print(f"Vocabulaire: {vocab_size} mots")
print(f"Données   : {len(data)} tokens")

embed_size = 50    # les vecteurs "évocateurs"
hidden_size = 100  # le nombre d'états cachés

model = LSTMNetwork(vocab_size, embed_size, hidden_size)
optimizer = SGDOptimizer(lr=0.1, decay_rate=0.01)
train_text(model, optimizer, data, seq_length=10, epochs=100)

print("\nGénération de texte:")
seed = data[:5]
generated = list(seed)
for _ in range(20):
    logits = model.forward_sequence(generated[-10:])
    next_token = int(np.argmax(logits[-1]))
    generated.append(next_token)

result = " ".join(idx_to_word.get(int(t), "?") for t in generated)
print(result)